In [3]:
# python
import datetime
import random
import time
from sqlalchemy import create_engine, Column, Integer, String, DateTime
from sqlalchemy.orm import declarative_base, sessionmaker

# 1. Database Configuration (Using SQLite for easy execution without complex local setups)
Base = declarative_base()
engine = create_engine('sqlite:///industrial_iot.db', echo=False)
Session = sessionmaker(bind=engine)
session = Session()

# 2. Database Model (Defining the Industrial Events Table)
class DowntimeLog(Base):
    __tablename__ = 'downtime_logs'
    
    id = Column(Integer, primary_key=True, autoincrement=True)
    machine_name = Column(String(50), nullable=False)
    error_code = Column(String(20), nullable=False)
    description = Column(String(250))
    timestamp = Column(DateTime, default=datetime.datetime.now)
    duration_minutes = Column(Integer)

# Create the table in the database
Base.metadata.create_all(engine)

# 3. Simulation Function (Simulating real-time PLC fault generation)
def simulate_plc_fault():
    machines = ["Robot_Kuka_L1", "Press_Hydraulic_04", "CNC_Milling_02", "Conveyor_Main_Assembly"]
    faults = [
        {"code": "E0402", "desc": "Emergency Stop Engaged - Safety Curtain Breached"},
        {"code": "E0115", "desc": "Motor Overheating - Thermal Overload"},
        {"code": "E0988", "desc": "Communication Timeout - Profinet Node Dropped"},
        {"code": "E0022", "desc": "Hydraulic Pressure Low - Fluid Leak Detected"}
    ]
    
    chosen_machine = random.choice(machines)
    chosen_fault = random.choice(faults)
    duration = random.randint(5, 120) # Simulated downtime duration in minutes
    
    # Create database registry object
    new_log = DowntimeLog(
        machine_name=chosen_machine,
        error_code=chosen_fault["code"],
        description=chosen_fault["desc"],
        duration_minutes=duration
    )
    
    # Save to SQL database
    session.add(new_log)
    session.commit()
    print(f"[SUCCESS] PLC Alert logged into Database: {chosen_machine} | Fault {chosen_fault['code']} | {duration} min")

# 4. Main Application Loop
if __name__ == "__main__":
    print("=== Starting Industrial IoT Data Pipeline Simulator ===")
    print("Press Ctrl+C to stop logging data.\n")
    
    try:
        for _ in range(5): # Generates 5 industrial logs automatically
            simulate_plc_fault()
            time.sleep(2) # Waits 2 seconds between logs
        print("\n[INFO] Simulation finished. Data successfully stored in 'industrial_iot.db'.")
    except KeyboardInterrupt:
        print("\n[INFO] Data logging stopped by user.")



=== Starting Industrial IoT Data Pipeline Simulator ===
Press Ctrl+C to stop logging data.

[SUCCESS] PLC Alert logged into Database: Conveyor_Main_Assembly | Fault E0988 | 31 min
[SUCCESS] PLC Alert logged into Database: Conveyor_Main_Assembly | Fault E0988 | 101 min
[SUCCESS] PLC Alert logged into Database: Conveyor_Main_Assembly | Fault E0022 | 31 min
[SUCCESS] PLC Alert logged into Database: Robot_Kuka_L1 | Fault E0988 | 68 min
[SUCCESS] PLC Alert logged into Database: Press_Hydraulic_04 | Fault E0402 | 95 min

[INFO] Simulation finished. Data successfully stored in 'industrial_iot.db'.
